In [1]:
import os
print("Current folder:", os.getcwd())
print("Files here:", os.listdir())

Current folder: /home/jovyan/notebooks
Files here: ['.ipynb_checkpoints', '.notremove', 'Untitled.ipynb', 'Untitled1.ipynb']


In [2]:
#Part 1: Load the Data
from pyspark.sql import SparkSession
print("PySpark import works")

PySpark import works


In [3]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

Spark 4.0.0-preview2 — ready


In [4]:
print("Files in current folder:", os.listdir())

if "data" in os.listdir():
    print("Files in data folder:", os.listdir("data"))

Files in current folder: ['.ipynb_checkpoints', '.notremove', 'Apache Spark.ipynb', 'Untitled.ipynb']


In [8]:
df = spark.read.json("data/transactions_10k.jsonl")

print(f"Record count: {df.count()}")
df.printSchema()
df.show(10, truncate=False)

Record count: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność 

In [9]:
#Part 2: Basic Aggregations
#Task 2.1 — Transaction count and total revenue per store
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp should now be 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [10]:
#Task 2.2 — Stats per category

from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
        _round(avg("amount"), 2).alias("avg_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+--------+----------+-------+
|   store|tx_count| total_PLN|avg_PLN|
+--------+--------+----------+-------+
|  Gdańsk|    2498|1021266.35| 408.83|
|  Kraków|    2522|1025896.95| 406.78|
|Warszawa|    2424| 961642.24| 396.72|
| Wrocław|    2556|1002739.21| 392.31|
+--------+--------+----------+-------+



In [12]:
#task 2.2
from pyspark.sql.functions import count, sum as _sum, min as _min, max as _max, round as _round

df.groupBy("category") \
  .agg(
      count("tx_id").alias("tx_count"),
      _round(_sum("amount"), 2).alias("total_PLN"),
      _round(_min("amount"), 2).alias("min_PLN"),
      _round(_max("amount"), 2).alias("max_PLN")
  ) \
  .orderBy("category") \
  .show()

+-----------+--------+----------+-------+-------+
|   category|tx_count| total_PLN|min_PLN|max_PLN|
+-----------+--------+----------+-------+-------+
|elektronika|    2542|1520770.69|    9.0| 9999.0|
|    książki|    2574| 851382.08|    5.0|9107.25|
|     odzież|    2453| 849877.55|    5.0|9696.63|
|    żywność|    2431| 789514.43|    5.0|6916.92|
+-----------+--------+----------+-------+-------+



In [13]:
#Task 3.1 — Transaction count per hour (tumbling 1h)

from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # 1-hour tumbling window
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+--------+----------+
|window                                    |tx_count|total_PLN |
+------------------------------------------+--------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150    |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661    |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189    |873403.24 |
+------------------------------------------+--------+----------+



In [14]:
(
    hourly
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+--------+----------+
|from               |to                 |tx_count|total_PLN |
+-------------------+-------------------+--------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150    |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661    |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189    |873403.24 |
+-------------------+-------------------+--------+----------+



In [15]:
#Task 3.2:  30-minute windows per store 
from pyspark.sql.functions import window, count, sum as _sum, round as _round, col

df.groupBy(window("timestamp", "30 minutes"), "store") \
  .agg(
      count("tx_id").alias("tx_count"),
      _round(_sum("amount"), 2).alias("total_PLN")
  ) \
  .select(
      col("window.start").alias("from"),
      col("window.end").alias("to"),
      "store",
      "tx_count",
      "total_PLN"
  ) \
  .orderBy("from", "store") \
  .show(truncate=False)

+-------------------+-------------------+--------+--------+---------+
|from               |to                 |store   |tx_count|total_PLN|
+-------------------+-------------------+--------+--------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252     |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289     |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275     |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296     |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514     |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532     |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490     |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502     |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619     |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590     |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584     |214573.66|
|2026-04-12 09:00:00

In [16]:
#Task 3.3:  Which hour had the highest revenue for store “Kraków”?
from pyspark.sql.functions import desc, window, count, sum as _sum, round as _round, col

df.filter(col("store") == "Kraków") \
  .groupBy(window("timestamp", "1 hour")) \
  .agg(
      count("tx_id").alias("tx_count"),
      _round(_sum("amount"), 2).alias("total_PLN")
  ) \
  .select(
      col("window.start").alias("from"),
      col("window.end").alias("to"),
      "tx_count",
      "total_PLN"
  ) \
  .orderBy(desc("total_PLN")) \
  .show(truncate=False)

+-------------------+-------------------+--------+---------+
|from               |to                 |tx_count|total_PLN|
+-------------------+-------------------+--------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|1169    |483309.86|
|2026-04-12 08:00:00|2026-04-12 09:00:00|821     |341327.83|
|2026-04-12 10:00:00|2026-04-12 11:00:00|532     |201259.26|
+-------------------+-------------------+--------+---------+



In [ ]:
#Part 4: Sliding Windows

In [17]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # 1h width, 30min step
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .orderBy("from")
)
sliding.show(truncate=False)

+-------------------+-------------------+--------+----------+
|from               |to                 |tx_count|total_PLN |
+-------------------+-------------------+--------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112    |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150    |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443    |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661    |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696    |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189    |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749     |289709.95 |
+-------------------+-------------------+--------+----------+



In [18]:
#Task 4.2 — Compare tumbling vs sliding
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):         {tumbling_rows} windows")
print(f"Sliding  (1h / 30min): {sliding_rows} windows")

# Answer in a comment: why does sliding produce more rows?
# YOUR ANSWER:
# Sliding produces more rows because the windows overlap.
# A new 1-hour window starts every 30 minutes, so Spark creates more time intervals
# than in tumbling windows, where windows are back-to-back and do not overlap.

Tumbling (1h):         3 windows
Sliding  (1h / 30min): 7 windows


In [ ]:
# Answer in comments:

# 1. How many transactions are in the 09:00–10:00 window?
#    Check the result from Task 3.1.
#    ANSWER: 4661 
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661    |1896230.21|

# 2. What is the difference between groupBy("store") and groupBy(window(...), "store")?
#    ANSWER:# groupBy("store") aggregates all transactions for each store across the whole dataset.
# groupBy(window(...), "store") aggregates transactions separately for each store inside each time window.

# 3. In the sliding window (1h / 30min), how many windows contain transactions from 09:30?
#    Hint: draw a timeline.
#    ANSWER: 3 windows contain transactions from 09:30:
# 08:30–09:30, 09:00–10:00, and 09:30–10:30.
2026-04-12 08:30:00|2026-04-12 09:30:00|4443    |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661    |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696    |1557641.39|

In [19]:
#Homework
#1.1 Calculate the average transaction amount and find out the hour when the dtore has lowest transactions

from pyspark.sql.functions import avg, round as _round, col, window

gdansk_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("avg_PLN")
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "avg_PLN"
    )
    .orderBy("avg_PLN")  
)

gdansk_avg.show(truncate=False)

+-------------------+-------------------+-------+
|from               |to                 |avg_PLN|
+-------------------+-------------------+-------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01 |
|2026-04-12 10:00:00|2026-04-12 11:00:00|412.92 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|415.91 |
+-------------------+-------------------+-------+



In [21]:
#30-minute windows per store
from pyspark.sql.functions import avg, round as _round, col, window

gdansk_avg_30min = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "30 minutes"))  
    .agg(
        _round(avg("amount"), 2).alias("avg_PLN")
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "avg_PLN"
    )
    .orderBy("avg_PLN")   # lowest first
)

gdansk_avg_30min.show(truncate=False)

+-------------------+-------------------+-------+
|from               |to                 |avg_PLN|
+-------------------+-------------------+-------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|370.6  |
|2026-04-12 10:30:00|2026-04-12 11:00:00|384.35 |
|2026-04-12 08:30:00|2026-04-12 09:00:00|406.98 |
|2026-04-12 09:00:00|2026-04-12 09:30:00|409.31 |
|2026-04-12 09:30:00|2026-04-12 10:00:00|423.27 |
|2026-04-12 10:00:00|2026-04-12 10:30:00|429.63 |
+-------------------+-------------------+-------+



In [ ]:
#Answer: From 8:00-9:00 store Gdańsk had the lowest average transaction amount.

In [22]:
#2. Count how many transactions per category occurred in the 09:00–09:30 window.

from pyspark.sql.functions import count

window_0900_0930 = (
    df.filter(
        (col("timestamp") >= "2026-04-12 09:00:00") &
        (col("timestamp") <  "2026-04-12 09:30:00")
    )
    .groupBy("category")
    .agg(count("tx_id").alias("tx_count"))
    .orderBy("category")
)

window_0900_0930.show()

+-----------+--------+
|   category|tx_count|
+-----------+--------+
|elektronika|     611|
|    książki|     622|
|     odzież|     605|
|    żywność|     567|
+-----------+--------+



In [23]:
#3. Use a 15-minute window and find which quarter-hour had the peak transaction volume (all stores combined).

from pyspark.sql.functions import desc

quarter_peak = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("tx_count"))
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count"
    )
    .orderBy(desc("tx_count"))   #show highest first
)

quarter_peak.show(truncate=False)

#Answer: From 9:15 - 9:30, there is highest number of transactions in all stores

+-------------------+-------------------+--------+
|from               |to                 |tx_count|
+-------------------+-------------------+--------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234    |
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171    |
|2026-04-12 09:30:00|2026-04-12 09:45:00|1156    |
|2026-04-12 08:45:00|2026-04-12 09:00:00|1139    |
|2026-04-12 09:45:00|2026-04-12 10:00:00|1100    |
|2026-04-12 08:30:00|2026-04-12 08:45:00|899     |
|2026-04-12 10:00:00|2026-04-12 10:15:00|858     |
|2026-04-12 08:15:00|2026-04-12 08:30:00|644     |
|2026-04-12 10:15:00|2026-04-12 10:30:00|582     |
|2026-04-12 08:00:00|2026-04-12 08:15:00|468     |
|2026-04-12 10:30:00|2026-04-12 10:45:00|443     |
|2026-04-12 10:45:00|2026-04-12 11:00:00|306     |
+-------------------+-------------------+--------+



In [ ]:
spark.stop()